####Requirement
Read data from flight_time and transform as below
1. Rename fl_date to dep_date
2. Compute arr_date
3. Following fields to represent full timestamp
    1. crs_dep_time
    2. dep_time
    3. crs_arr_time
    4. arr_time



In [0]:
%sql
select * From dev.spark_db.flight_time;

1. Can we do it using select() or selectExpr() transformations?

In [0]:
flight_time_df = spark.table("dev.spark_db.flight_time")

flight_time_df_1 = (
    flight_time_df.selectExpr(
        "fl_date as dep_date",
        "to_date(dep_date + dep_time + wheels_on + taxi_in) as arr_date",
        "dep_date + crs_dep_time as crs_dep_time",
        "dep_date + dep_time as dep_time",
        "arr_date + crs_arr_time as crs_arr_time",
        "arr_date + arr_time as arr_time",
        "op_carrier"
    )
)
#SelectExpr is not good transformation because you will lost the columns better to use withColumns if all the columns are required. if you need particular columns only you can use selectexpr

flight_time_df_1.where("op_carrier_fl_num = 1451 and dep_date = '2000-01-01'").display()

2. Can we do it using withColumn() or withColumns()?

In [0]:
from pyspark.sql.functions import expr,col

flight_time_df2 = (
  flight_time_df.withColumnRenamed("fl_date", "dep_date")
      .withColumn("arr_date", expr("to_date(dep_date + dep_time + wheels_on + taxi_in) as arr_date"))
      .withColumns({
        "crs_dep_time": expr("dep_date + crs_dep_time"),
        "dep_time": expr("dep_date + dep_time"),
        "crs_arr_time": expr("arr_date + crs_arr_time"),
        "arr_time": col("arr_date") + col("arr_time"), #col - You can use col to access the column (This is other way to write expr)
      })
)

flight_time_df2.where("op_carrier_fl_num = 1451 and dep_date = '2000-01-01'").display()

3. Alternative approach to write expressions\
  Why to use it?
    * It gives access to column functions

In [0]:
from pyspark.sql.functions import to_date, col

flight_time_df3 = (
  flight_time_df.withColumnRenamed("fl_date", "dep_date")
      .withColumn("arr_date", to_date(col("dep_date") + col("dep_time") + col("wheels_on") + col("taxi_in")))
      .withColumns({
        "crs_dep_time": col("dep_date") + col("crs_dep_time"),
        "dep_time": col("dep_date") + col("dep_time"),
        "crs_arr_time": col("arr_date") + col("crs_arr_time"),
        "arr_time": col("arr_date") + col("arr_time"),
      })
)

flight_time_df3.where((col("op_carrier_fl_num") == 1451) & (col("dep_date") == '2000-01-01')).display()